# `dutch_railway / eda`

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.environ.setdefault("DUTCH_RAILWAY_DATA_DIR", str(ROOT / "data"))
sys.path.insert(0, str(ROOT / "dags"))

from etl import config, extract
import duckdb

remote = extract.fetch_remote_metadata(config.source_url(2025))
RAW = extract.download(remote, config.raw_path(2025))  # raw CSV: header + sniffer checks
STAGED = config.staged_path(2025)                      # typed Parquet from the extract (run the DAG first)
remote

RemoteFile(url='https://blobs.duckdb.org/nl-railway/services-2025.csv.gz', size=396222719, etag='b37847417628434f788208305b25a11d-48', last_modified='Wed, 14 Jan 2026 15:45:16 GMT')

## Overview

One tuple is **one stop of one train service**. The `Service:*` columns describe the whole trip and repeat on every stop of it; the `Stop:*` columns describe that one stop.

In [2]:
con = duckdb.connect()
con.execute(f"SET TimeZone = '{config.SOURCE_TIMEZONE}'")
con.execute(f"CREATE VIEW raw AS FROM read_csv('{RAW}')")            
con.execute(f"CREATE VIEW staged AS FROM read_parquet('{STAGED}')")  

rows = con.sql("SELECT count(*) FROM staged").fetchone()[0]
print(f"raw CSV:        {RAW.stat().st_size / 1e6:.0f} MB (gzip)")
print(f"staged Parquet: {STAGED.stat().st_size / 1e6:.0f} MB")
print(f"shape:          {rows:,} rows x {len(config.SERVICES_COLUMNS)} columns")

raw CSV:        396 MB (gzip)
staged Parquet: 193 MB
shape:          22,226,311 rows x 20 columns


### Columns

In [3]:
import pandas as pd

cols = list(config.SERVICES_COLUMNS)
aggs = ", ".join(f'count("{c}"), count(DISTINCT "{c}"), min("{c}")::VARCHAR, max("{c}")::VARCHAR' for c in cols)
stats = con.execute(f"SELECT {aggs} FROM staged").fetchone()  # one pass over the file
sniffed = dict(con.execute("SELECT column_name, column_type FROM (DESCRIBE raw)").fetchall())
staged_types = dict(con.execute("SELECT column_name, column_type FROM (DESCRIBE staged)").fetchall())

pd.DataFrame(
    [
        {
            "column": c,
            "sniffed (CSV)": sniffed[c],
            "staged": staged_types[c],
            "non_null": stats[4 * i],
            "null_%": round(100 * (1 - stats[4 * i] / rows), 2),
            "distinct": stats[4 * i + 1],
            "min": stats[4 * i + 2],
            "max": stats[4 * i + 3],
        }
        for i, c in enumerate(cols)
    ]
).set_index("column")

,sniffed (CSV),staged,non_null,null_%,distinct,min,max
column,,,,,,,
Service:RDT-ID,BIGINT,BIGINT,22226311,0.00,2402665,15086865,17504617
Service:Date,DATE,DATE,22226311,0.00,365,2025-01-01,2025-12-31
Service:Type,VARCHAR,VARCHAR,22226311,0.00,21,Belbus ipv trein,Taxibus ipv trein
Service:Company,VARCHAR,VARCHAR,22226311,0.00,32,137,VIAS
Service:Train number,BIGINT,BIGINT,22226311,0.00,35317,25,973063
Service:Completely cancelled,BOOLEAN,BOOLEAN,22226311,0.00,2,false,true
Service:Partly cancelled,BOOLEAN,BOOLEAN,22226311,0.00,2,false,true
Service:Maximum delay,BIGINT,BIGINT,22226311,0.00,227,0,1440
Stop:RDT-ID,BIGINT,BIGINT,22226311,0.00,22226311,136167814,158545506


### Sample rows

Five random rows (fixed seed, so the output is stable), transposed so all 20 columns fit.

In [4]:
con.sql("SELECT * FROM staged USING SAMPLE reservoir(5 ROWS) REPEATABLE (42)").df().T

,0,1,2,3,4
Service:RDT-ID,15092529,15092615,15093960,15094221,15094415
Service:Date,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00,2025-01-02 00:00:00
Service:Type,Sprinter,Sprinter,Stoptrein,Sprinter,Sneltrein
Service:Company,NS,NS,Arriva,NS,Eurobahn
Service:Train number,7325,6931,30833,9043,20080
Service:Completely cancelled,False,False,False,False,False
Service:Partly cancelled,False,False,False,False,False
Service:Maximum delay,0,0,0,2,0
Stop:RDT-ID,136222361,136223249,136236955,136239535,136241562
Stop:Station code,UTVR,UTLN,ZP,LLS,DUSSEL


### One service, stop by stop

`Service:*` values repeat, the first stop has no arrival and the last stop has no departure (that is where the ~10.8% NULLs come from).

In [5]:
con.sql("""
    SELECT "Stop:Station name", "Stop:Arrival time", "Stop:Arrival delay", "Stop:Departure time",
           "Stop:Departure delay", "Stop:Arrival cancelled", "Stop:Departure cancelled", "Stop:Planned platform"
    FROM staged
    WHERE "Service:RDT-ID" = (
        SELECT min("Service:RDT-ID") FROM staged
        WHERE "Service:Type" = 'Intercity' AND "Service:Date" = DATE '2025-03-12'
    )
    ORDER BY coalesce("Stop:Arrival time", "Stop:Departure time")
""")

┌─────────────────────────┬──────────────────────────┬────────────────────┬──────────────────────────┬──────────────────────┬────────────────────────┬──────────────────────────┬───────────────────────┐
│    Stop:Station name    │    Stop:Arrival time     │ Stop:Arrival delay │   Stop:Departure time    │ Stop:Departure delay │ Stop:Arrival cancelled │ Stop:Departure cancelled │ Stop:Planned platform │
│         varchar         │ timestamp with time zone │       int64        │ timestamp with time zone │        int64         │        boolean         │         boolean          │        varchar        │
├─────────────────────────┼──────────────────────────┼────────────────────┼──────────────────────────┼──────────────────────┼────────────────────────┼──────────────────────────┼───────────────────────┤
│ Utrecht Centraal        │ NULL                     │               NULL │ 2025-03-12 02:16:00+01   │                    0 │ NULL                   │ false                    │ 18            

## Null checks

Only 8 of the 20 columns have NULLs (see the Columns table). They fall into three groups. If every column in a group is always NULL together, the NULLs are probably **structural** (the value cannot exist) rather than lost data.

In [6]:
groups = {
    "arrival":   ["Stop:Arrival time", "Stop:Arrival delay", "Stop:Arrival cancelled"],
    "departure": ["Stop:Departure time", "Stop:Departure delay", "Stop:Departure cancelled"],
    "platform":  ["Stop:Planned platform", "Stop:Actual platform"],
}
con.sql(" UNION ALL ".join(
    f"""
    SELECT '{g}' AS "group",
           count(*) FILTER ({" AND ".join(f'"{c}" IS NULL' for c in cs)})     AS all_null,
           count(*) FILTER ({" AND ".join(f'"{c}" IS NOT NULL' for c in cs)}) AS none_null,
           count(*) - all_null - none_null                                      AS mixed
    FROM staged"""
    for g, cs in groups.items()
))

┌───────────┬──────────┬───────────┬───────┐
│   group   │ all_null │ none_null │ mixed │
│  varchar  │  int64   │   int64   │ int64 │
├───────────┼──────────┼───────────┼───────┤
│ arrival   │  2403837 │  19822474 │     0 │
│ departure │  2404493 │  19821818 │     0 │
│ platform  │  3008309 │  19218002 │     0 │
└───────────┴──────────┴───────────┴───────┘

**Remarks**:
- `mixed = 0` for all three groups: a stop either has the full arrival record (time, delay, cancelled flag) or none of it. The same holds for departure and for the two platform columns.
- So each group can be read as one yes/no question: *does this stop have an arrival / a departure / a platform?*

### Arrival and departure NULLs: start and end of a trip

Hypothesis: a train has no arrival at its first stop and no departure at its last stop. Order each service's stops by time and check where the NULLs fall.

In [7]:
con.sql("""
    WITH stops AS (
        SELECT "Stop:Arrival time" IS NULL   AS no_arrival,
               "Stop:Departure time" IS NULL AS no_departure,
               row_number() OVER w           AS position,
               count(*) OVER (PARTITION BY "Service:RDT-ID") AS n_stops
        FROM staged
        WINDOW w AS (PARTITION BY "Service:RDT-ID"
                     ORDER BY coalesce("Stop:Arrival time", "Stop:Departure time"), "Stop:RDT-ID")
    )
    SELECT count(*) FILTER (no_arrival)                          AS no_arrival,
           count(*) FILTER (no_arrival AND position = 1)         AS at_first_stop,
           count(*) FILTER (no_departure)                        AS no_departure,
           count(*) FILTER (no_departure AND position = n_stops) AS at_last_stop
    FROM stops
""")

┌────────────┬───────────────┬──────────────┬──────────────┐
│ no_arrival │ at_first_stop │ no_departure │ at_last_stop │
│   int64    │     int64     │    int64     │    int64     │
├────────────┼───────────────┼──────────────┼──────────────┤
│    2403837 │       2402462 │      2404493 │      2402280 │
└────────────┴───────────────┴──────────────┴──────────────┘

**Remarks**:
- The hypothesis holds for 99.9% of the NULLs. Missing arrival means **origin** of the trip, missing departure means **terminus**.
- These are not missing data, so they should not be filled in or dropped.
- A small remainder (about 1.4k arrivals and 2.2k departures) are NULL somewhere in the middle of a trip. Possible explanations, not yet checked: trains that are split or joined along the way, or stops in the same minute where ordering by time is ambiguous.

### Platform NULLs

13.5% of stops have no platform. Split them by what kind of stop it is:

In [8]:
con.sql(f"""
    SELECT CASE WHEN ("Service:Type" LIKE '% ipv trein' OR "Service:Type" = 'Bus') THEN 'bus / metro'
                WHEN (coalesce("Stop:Arrival cancelled", true) AND coalesce("Stop:Departure cancelled", true)) THEN 'train, stop cancelled'
                ELSE 'train, stop served' END                        AS kind,
           count(*)                                                   AS stops,
           round(100 * avg(("Stop:Planned platform" IS NULL)::INT), 1) AS pct_no_platform
    FROM staged GROUP BY 1 ORDER BY stops DESC
""")

┌───────────────────────┬──────────┬─────────────────┐
│         kind          │  stops   │ pct_no_platform │
│        varchar        │  int64   │     double      │
├───────────────────────┼──────────┼─────────────────┤
│ train, stop served    │ 20320388 │             5.4 │
│ train, stop cancelled │  1152613 │           100.0 │
│ bus / metro           │   753310 │           100.0 │
└───────────────────────┴──────────┴─────────────────┘

In [ ]:
con.sql(f"""
    SELECT "Service:Company" AS operator, count(*) AS stops,
           round(100 * avg(("Stop:Planned platform" IS NULL)::INT), 1) AS pct_no_platform
    FROM staged WHERE NOT ("Service:Type" LIKE '% ipv trein' OR "Service:Type" = 'Bus') AND NOT (coalesce("Stop:Arrival cancelled", true) AND coalesce("Stop:Departure cancelled", true))
    GROUP BY 1 ORDER BY stops DESC LIMIT 12
""")

┌──────────┬──────────┬─────────────────┐
│ operator │  stops   │ pct_no_platform │
│ varchar  │  int64   │     double      │
├──────────┼──────────┼─────────────────┤
│ NS       │ 13349551 │             0.2 │
│ Arriva   │  3587212 │             2.6 │
│ Blauwnet │   951922 │             0.5 │
│ R-net    │   652426 │             0.0 │
│ DB       │   364367 │            82.2 │
│ NS Int   │   349211 │            59.6 │
│ RRReis   │   278060 │             0.0 │
│ Eurobahn │   195544 │            87.1 │
│ NMBS     │   176498 │            93.4 │
│ Breng    │   118608 │             0.0 │
│ VIAS     │   102085 │            85.0 │
│ TCS OV   │    31185 │            91.3 │
└──────────┴──────────┴─────────────────┘
  12 rows                     3 columns

**Remarks**:
- **Buses and metros** never have a platform (100%): they stop at a bus stop, not a track. Structural.
- **Cancelled stops** never have one (100%): the train did not come, so no track was assigned. Structural.
- **Foreign operators** (DB, NMBS, Eurobahn, VIAS, TCS OV, and NS International abroad) leave it empty 60-93% of the time, while Dutch operators are below 3%. Likely the source (Dutch rail data) only knows platforms at Dutch stations. This one is **real missing data**, not structural.
- So 'no platform' must not be read as 'platform unknown' in general: for buses and cancelled stops there is nothing to know.

A `NULL` is truly being unknown. A `0` that means 'unknown' enters averages or numerical computions. 

Check delays of cancelled stops, and service types that never report a delay.

In [10]:
con.sql("""
    SELECT "Stop:Arrival cancelled" AS arrival_cancelled, count(*) AS stops,
           min("Stop:Arrival delay") AS min_delay, max("Stop:Arrival delay") AS max_delay
    FROM staged GROUP BY 1 ORDER BY 1
""")

┌───────────────────┬──────────┬───────────┬───────────┐
│ arrival_cancelled │  stops   │ min_delay │ max_delay │
│      boolean      │  int64   │   int64   │   int64   │
├───────────────────┼──────────┼───────────┼───────────┤
│ false             │ 18707153 │         0 │      1440 │
│ true              │  1115321 │         0 │         0 │
│ NULL              │  2403837 │      NULL │      NULL │
└───────────────────┴──────────┴───────────┴───────────┘

In [11]:
# service types / operators where the delay is never anything but 0
con.sql("""
    SELECT "Service:Type" AS type, "Service:Company" AS operator, count(*) AS stops,
           max("Stop:Arrival delay") AS max_arrival_delay, max("Stop:Departure delay") AS max_departure_delay
    FROM staged GROUP BY ALL
    HAVING max("Stop:Arrival delay") = 0 AND max("Stop:Departure delay") = 0
    ORDER BY stops DESC
""")

┌───────────────────┬────────────┬────────┬───────────────────┬─────────────────────┐
│       type        │  operator  │ stops  │ max_arrival_delay │ max_departure_delay │
│      varchar      │  varchar   │ int64  │       int64       │        int64        │
├───────────────────┼────────────┼────────┼───────────────────┼─────────────────────┤
│ Stopbus ipv trein │ Arriva     │ 156455 │                 0 │                   0 │
│ Metro ipv trein   │ NS         │  88233 │                 0 │                   0 │
│ Stopbus ipv trein │ VIAS       │  28484 │                 0 │                   0 │
│ Snelbus ipv trein │ Arriva     │  28068 │                 0 │                   0 │
│ Stopbus ipv trein │ Blauwnet   │  21827 │                 0 │                   0 │
│ Stoptrein         │ Deutsche B │  20480 │                 0 │                   0 │
│ Taxibus ipv trein │ NS         │  13227 │                 0 │                   0 │
│ Snelbus ipv trein │ VIAS       │  11339 │           

**Remarks**:
- **Cancelled stops always have delay 0** (1.1M stops). A train that did not come was not on time. Averaging these in makes trains look more punctual than they were.
- **Some services never report a delay**: metro/taxi/OV-bus/belbus replacements, the `Bus` type, and Deutsche Bahn under its December name `Deutsche B`. Over a whole year, 'never late once' is not believable; more likely there is no real-time tracking for them, and 0 is a default.
- Maybe note for transform: store these delays as NULL so `AVG(delay)` only covers stops where a delay was actually measured.

## Value sets

List every value with its share, and what it likely means.

### Boolean flags

In [12]:
flags = ["Service:Completely cancelled", "Service:Partly cancelled",
         "Stop:Arrival cancelled", "Stop:Departure cancelled", "Stop:Platform change"]
con.sql(" UNION ALL ".join(
    f"""
    SELECT '{c}' AS flag,
           count(*) FILTER ("{c}")         AS "true",
           count(*) FILTER (NOT "{c}")     AS "false",
           count(*) FILTER ("{c}" IS NULL) AS "null",
           round(100 * count(*) FILTER ("{c}") / count("{c}"), 2) AS pct_true
    FROM staged"""
    for c in flags
))

┌──────────────────────────────┬─────────┬──────────┬─────────┬──────────┐
│             flag             │  true   │  false   │  null   │ pct_true │
│           varchar            │  int64  │  int64   │  int64  │  double  │
├──────────────────────────────┼─────────┼──────────┼─────────┼──────────┤
│ Service:Completely cancelled │  425577 │ 21800734 │       0 │     1.91 │
│ Service:Partly cancelled     │ 2178957 │ 20047354 │       0 │      9.8 │
│ Stop:Arrival cancelled       │ 1115321 │ 18707153 │ 2403837 │     5.63 │
│ Stop:Departure cancelled     │ 1114776 │ 18707042 │ 2404493 │     5.62 │
│ Stop:Platform change         │  639241 │ 21587070 │       0 │     2.88 │
└──────────────────────────────┴─────────┴──────────┴─────────┴──────────┘

**Remarks**:
- Possible interpretation of each flag:

| flag | level | `true` means |
|---|---|---|
| `Service:Completely cancelled` | whole trip, repeated on every stop | the trip did not run at all |
| `Service:Partly cancelled` | whole trip, repeated on every stop | at least one stop of the trip was skipped |
| `Stop:Arrival cancelled` | one stop | the train did not arrive here (NULL = origin, see above) |
| `Stop:Departure cancelled` | one stop | the train did not leave from here (NULL = terminus) |
| `Stop:Platform change` | one stop | the train used a different track than planned |

- The `Service:*` percentages count stops, not trips: a cancelled 10-stop trip counts 10 times. Below, the two service flags are checked against what actually happened at the stops.

In [13]:
# is Platform change just "planned platform != actual platform"?
con.sql("""
    SELECT "Stop:Platform change" AS platform_change,
           "Stop:Planned platform" IS DISTINCT FROM "Stop:Actual platform" AS platforms_differ,
           count(*) AS stops
    FROM staged GROUP BY ALL ORDER BY ALL
""")

┌─────────────────┬──────────────────┬──────────┐
│ platform_change │ platforms_differ │  stops   │
│     boolean     │     boolean      │  int64   │
├─────────────────┼──────────────────┼──────────┤
│ false           │ false            │ 21587070 │
│ true            │ true             │   639241 │
└─────────────────┴──────────────────┴──────────┘

In [14]:
# one row per trip: do the service-level flags match the stop-level cancellations?
con.sql(f"""
    WITH trips AS (
        SELECT any_value("Service:Completely cancelled") AS completely,
               any_value("Service:Partly cancelled")     AS partly,
               count(*)                                  AS stops,
               count(*) FILTER ((coalesce("Stop:Arrival cancelled", true) AND coalesce("Stop:Departure cancelled", true)))       AS cancelled_stops
        FROM staged GROUP BY "Service:RDT-ID"
    )
    SELECT completely, partly, count(*) AS trips,
           count(*) FILTER (cancelled_stops = 0)                      AS no_stop_cancelled,
           count(*) FILTER (cancelled_stops BETWEEN 1 AND stops - 1)  AS some_stops_cancelled,
           count(*) FILTER (cancelled_stops = stops)                  AS all_stops_cancelled
    FROM trips GROUP BY ALL ORDER BY ALL
""")

┌────────────┬─────────┬─────────┬───────────────────┬──────────────────────┬─────────────────────┐
│ completely │ partly  │  trips  │ no_stop_cancelled │ some_stops_cancelled │ all_stops_cancelled │
│  boolean   │ boolean │  int64  │       int64       │        int64         │        int64        │
├────────────┼─────────┼─────────┼───────────────────┼──────────────────────┼─────────────────────┤
│ false      │ false   │ 2213845 │           2213807 │                   38 │                   0 │
│ false      │ true    │  136722 │              1095 │               135627 │                   0 │
│ true       │ false   │    1836 │              1836 │                    0 │                   0 │
│ true       │ true    │   50262 │                 0 │                  372 │               49890 │
└────────────┴─────────┴─────────┴───────────────────┴──────────────────────┴─────────────────────┘

**Remarks**:
- `Stop:Platform change` is exactly `planned != actual` (no exceptions), so it is redundant: it can be derived from the two platform columns.
- The two service flags are **not** two levels of one scale:
  - `partly = true, completely = false` (136.7k trips): some stops skipped, the rest served. Matches the name.
  - `completely = true` almost always comes **with** `partly = true` (50.3k trips), and then all stops are cancelled in 49.9k of them (372 still served some stops). So 'completely' looks like an extra flag on top of 'partly', not an alternative to it.
  - `completely = true, partly = false` (1,836 trips) has **no** cancelled stops at all. Possible meaning: the trip was cancelled and replaced by another service (e.g. a bus listed separately), or a data inconsistency.
  - Smaller mismatches: 1,095 trips flagged `partly` with no cancelled stop, and 38 trips with cancelled stops but neither flag.
- For counting cancellations, the stop-level flags look more trustworthy than the service-level ones.

### `Service:Type` (21 values)

In [15]:
con.sql("""
    SELECT "Service:Type" AS type, count(DISTINCT "Service:RDT-ID") AS trips, count(*) AS stops,
           round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct_stops
    FROM staged GROUP BY 1 ORDER BY stops DESC
""").df()

,type,trips,stops,pct_stops
0,Sprinter,977485,10378405,46.69
1,Stoptrein,536385,5093009,22.91
2,Intercity,487740,4629722,20.83
3,Sneltrein,90460,754884,3.40
4,Stopbus ipv trein,111252,474438,2.13
5,Intercity direct,41235,271723,1.22
6,Snelbus ipv trein,76096,174902,0.79
7,EuroCity,11574,112666,0.51
8,Metro ipv trein,32263,88233,0.40
9,Eurocity Direct,11528,74201,0.33


**Remarks**:
- The values are Dutch; *ipv* = *in plaats van* = *instead of*. Possible interpretation:

| value | meaning | group |
|---|---|---|
| Sprinter | NS local train, calls at every station | domestic train |
| Stoptrein | 'stopping train': local train of a regional operator (the non-NS word for a Sprinter) | domestic train |
| Intercity | fast train between larger stations | domestic train |
| Sneltrein | 'fast train': regional train that skips small stations | domestic train |
| Intercity direct | NS high-speed Intercity over the HSL-Zuid line (Amsterdam - Rotterdam - Breda) | domestic train |
| Nachttrein | night train | domestic train |
| EuroCity, Eurocity Direct | international trains (Eurocity Direct: Amsterdam - Brussels) | international train |
| Eurostar | high-speed train to Brussels / Paris / London | international train |
| ICE, ICE International | German high-speed trains | international train |
| Nightjet, European Sleeper | international night trains | international train |
| Int. Trein | other international train | international train |
| Stopbus / Snelbus ipv trein | bus calling at every station / express bus, instead of a train | replacement |
| Metro ipv trein | metro instead of a train | replacement |
| Taxibus / OV-bus / Belbus ipv trein | taxi, regular public bus, or on-demand bus instead of a train | replacement |
| Bus | a bus that is part of the timetable (R-net only) | bus |

- Replacement services (`... ipv trein`) run during engineering works or disruptions.
- Whether they count as 'services' (for delay averages, for example) is a design decision: they have no platforms and mostly no delay data (see Null checks).

### `Service:Company` (32 values)

In [16]:
con.sql("""
    SELECT "Service:Company" AS operator, count(DISTINCT "Service:RDT-ID") AS trips,
           min("Service:Date") AS first_day, max("Service:Date") AS last_day,
           count(DISTINCT "Service:Date") AS days,
           string_agg(DISTINCT "Service:Type", ', ') AS types
    FROM staged GROUP BY 1 ORDER BY trips DESC
""").df()

,operator,trips,first_day,last_day,days,types
0,NS,1509432,2025-01-01,2025-12-31,365,"Snelbus ipv trein, Intercity, Taxibus ipv trei..."
1,Arriva,462281,2025-01-01,2025-12-31,365,"Sneltrein, Stopbus ipv trein, Nachttrein, Stop..."
2,Blauwnet,138373,2025-01-01,2025-12-13,347,"Stoptrein, Intercity, Taxibus ipv trein, Snelb..."
3,R-net,79760,2025-01-01,2025-12-13,347,"Stopbus ipv trein, Sprinter, Stoptrein, Bus, S..."
4,NS Int,45435,2025-01-01,2025-12-31,361,"Intercity, Eurocity Direct, Eurostar, Snelbus ..."
5,RRReis,40969,2025-01-01,2025-12-13,347,"Sprinter, Stopbus ipv trein"
6,Eurobahn,32338,2025-01-01,2025-12-31,365,"Stoptrein, Snelbus ipv trein, Sneltrein, Stopb..."
7,DB,25729,2025-01-01,2025-12-13,347,"Stoptrein, Stopbus ipv trein"
8,VIAS,18816,2025-01-01,2025-12-31,365,"Snelbus ipv trein, Belbus ipv trein, Stoptrein..."
9,Breng,13431,2025-01-02,2025-12-12,245,Stoptrein


**Remarks**:
- The `first_day` / `last_day` columns show that the operator names **change format on 14 December 2025**, the day of the yearly timetable change.
- **Names are cut to 10 characters**: `Deutsche B` = Deutsche Bahn, `Eu Sleeper` = European Sleeper.
- **The same operator gets a new name in December**: `DB` runs until 13 Dec, `Deutsche B` from 14 Dec.
- **Numeric codes** (`63`, `69`, `137`, `154`, `155`, `970`, ...) appear almost only on **14-18 December**. They run on the same routes as named operators (for example `155` runs Zwolle - Enschede like `Blauwnet K`, and `63` runs Amersfoort - Barneveld like `RRReis`). Possible meaning: operator IDs whose name was missing for the first days of the new timetable.
- **Suffixed names** (`Blauwnet A`/`Blauwnet K`, `RRReis A`/`RRReis K`, `R-net NS`/`R-net Qb`) start on 18 December. Possibly brand + operator (A = Arriva, K = Keolis, Qb = Qbuzz), after regional contracts changed hands. Not confirmed.
- Consequence for the schema: `Service:Company` is not a clean operator key. To compare operators across the whole year, names need a mapping (e.g. `DB` and `Deutsche B` -> Deutsche Bahn), and the numeric codes need either a mapping by route or an 'unknown operator' label.

### Platforms (51 values)

In [17]:
con.sql("""
    SELECT CASE WHEN regexp_full_match(p, '[0-9]+')      THEN 'number'
                WHEN regexp_full_match(p, '[0-9]+[ab]')  THEN 'number + a/b'
                ELSE 'other' END                            AS pattern,
           count(DISTINCT p) AS n_values, count(*) AS stops,
           string_agg(DISTINCT p, ', ' ORDER BY p) AS values
    FROM (SELECT "Stop:Planned platform" AS p FROM staged WHERE p IS NOT NULL)
    GROUP BY 1 ORDER BY stops DESC
""").df()

,pattern,n_values,stops,values
0,number,21,17201979,"1, 10, 11, 12, 13, 14, 15, 16, 18, 19, 2, 20, ..."
1,number + a/b,28,1843726,"10a, 10b, 11a, 11b, 13a, 13b, 14a, 14b, 15a, 1..."
2,other,2,172297,"1/2, 5/6"


**Remarks**:
- Platform is a **label, not a number**: `4b` is the second half of a long platform 4, and `1/2` / `5/6` probably mean the train stands across two tracks.
- That is why it stays text. Sorting it as text puts `10` before `2`, so sort by the numeric part if needed.

### Delays

In [18]:
con.sql("""
    SELECT CASE WHEN d IS NULL THEN 'NULL (origin)'
                WHEN d = 0     THEN '0'
                WHEN d < 3     THEN '1-2'
                WHEN d < 5     THEN '3-4'
                WHEN d < 15    THEN '5-14'
                WHEN d < 30    THEN '15-29'
                WHEN d < 60    THEN '30-59'
                WHEN d < 1440  THEN '60-1439'
                ELSE '1440' END                        AS arrival_delay_min,
           count(*)                                       AS stops,
           round(100 * count(*) / sum(count(*)) OVER (), 2) AS pct
    FROM (SELECT "Stop:Arrival delay" AS d FROM staged)
    GROUP BY 1 ORDER BY min(d) NULLS FIRST
""")

┌───────────────────┬──────────┬────────┐
│ arrival_delay_min │  stops   │  pct   │
│      varchar      │  int64   │ double │
├───────────────────┼──────────┼────────┤
│ NULL (origin)     │  2403837 │  10.82 │
│ 0                 │ 14158347 │   63.7 │
│ 1-2               │  3927149 │  17.67 │
│ 3-4               │   863561 │   3.89 │
│ 5-14              │   720656 │   3.24 │
│ 15-29             │   119043 │   0.54 │
│ 30-59             │    26772 │   0.12 │
│ 60-1439           │     6944 │   0.03 │
│ 1440              │        2 │    0.0 │
└───────────────────┴──────────┴────────┘

In [19]:
# is Service:Maximum delay the largest stop delay of the trip?
con.sql("""
    WITH trips AS (
        SELECT any_value("Service:Maximum delay") AS reported,
               max(greatest(coalesce("Stop:Arrival delay", 0), coalesce("Stop:Departure delay", 0))) AS computed
        FROM staged GROUP BY "Service:RDT-ID"
    )
    SELECT count(*) AS trips,
           count(*) FILTER (reported = computed) AS equal,
           count(*) FILTER (reported > computed) AS reported_higher,
           count(*) FILTER (reported < computed) AS reported_lower
    FROM trips
""")

┌─────────┬─────────┬─────────────────┬────────────────┐
│  trips  │  equal  │ reported_higher │ reported_lower │
│  int64  │  int64  │      int64      │     int64      │
├─────────┼─────────┼─────────────────┼────────────────┤
│ 2402665 │ 2401643 │             487 │            535 │
└─────────┴─────────┴─────────────────┴────────────────┘

**Remarks**:
- Delays are whole **minutes**, never negative (early trains count as 0). Most arrivals are exactly on time, and the distribution has a long tail. The single `1440` (exactly 24 hours) on one trip is most likely a cap or placeholder, not a real delay.
- `Service:Maximum delay` equals the largest stop delay of the trip for almost every trip, so it is derived data. A small number of trips disagree in both directions; a possible cause is a delay recorded on an event that is not in this file.